# NLP Communication AnalysisConverted from `src/nlp_analysis.py`---

**Beschreibung:** NLP Analysis - Sentiment and communication quality

In [1]:
import pandas as pdimport numpy as npimport refrom pathlib import Path# VADER Sentiment (optional import)try:    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer    VADER_AVAILABLE = Trueexcept ImportError:    VADER_AVAILABLE = False    print(" VADER nicht installiert. Installiere mit: pip install vaderSentiment")# Word lists for pattern recognitionPOLITENESS_WORDS = [    'please', 'thank', 'thanks', 'appreciate', 'grateful',    'sorry', 'apolog', 'kindly', 'would you', 'could you']URGENCY_WORDS = [    'urgent', 'asap', 'immediately', 'critical', 'emergency',    'deadline', 'priority', 'important', 'blocker', 'blocked']TECHNICAL_WORDS = [    'error', 'bug', 'fix', 'issue', 'problem', 'crash',    'exception', 'failed', 'timeout', 'null', 'undefined']SOLUTION_WORDS = [    'fixed', 'resolved', 'solved', 'solution', 'working',    'deployed', 'released', 'updated', 'patched', 'done']def analyze_sentiment(text):    """    Analyze das Sentiment eines Textes.    Returns:        dict: Sentiment-Scores (compound, pos, neg, neu)    """    if not VADER_AVAILABLE:        return {'compound': 0, 'pos': 0, 'neg': 0, 'neu': 1}    if pd.isna(text) or not str(text).strip():        return {'compound': 0, 'pos': 0, 'neg': 0, 'neu': 1}    analyzer = SentimentIntensityAnalyzer()    return analyzer.polarity_scores(str(text))def extract_text_features(text):    """Extrahiert Basis-Features aus Text."""    if pd.isna(text) or not str(text).strip():        return {            'word_count': 0,            'char_count': 0,            'question_count': 0,            'exclamation_count': 0        }    text = str(text)    words = text.split()    return {        'word_count': len(words),        'char_count': len(text),        'question_count': text.count('?'),        'exclamation_count': text.count('!')    }def extract_patterns(text):    """Count communication patterns in text."""    if pd.isna(text) or not str(text).strip():        return {            'politeness_score': 0,            'urgency_score': 0,            'technical_score': 0,            'solution_score': 0        }    text_lower = str(text).lower()    return {        'politeness_score': sum(1 for w in POLITENESS_WORDS if w in text_lower),        'urgency_score': sum(1 for w in URGENCY_WORDS if w in text_lower),        'technical_score': sum(1 for w in TECHNICAL_WORDS if w in text_lower),        'solution_score': sum(1 for w in SOLUTION_WORDS if w in text_lower)    }def process_utterances(utterances_df):    """    Verarbeitet alle Utterances und extrahiert NLP-Features.    Args:        utterances_df: DataFrame mit Kommentaren    Returns:        DataFrame mit NLP-Features    """    print(" Verarbeite Kommentare...")    # Text-Spalte finden    text_col = 'actionbody' if 'actionbody' in utterances_df.columns else 'body'    results = []    total = len(utterances_df)    for idx, row in utterances_df.iterrows():        text = row[text_col] if text_col in row else ""        # Sentiment        sentiment = analyze_sentiment(text)        # Text-Features        text_feat = extract_text_features(text)        # Patterns        patterns = extract_patterns(text)        results.append({            'issueid': row.get('issueid', idx),            'author_role': row.get('author_role', 'unknown'),            'sentiment_compound': sentiment['compound'],            'sentiment_pos': sentiment['pos'],            'sentiment_neg': sentiment['neg'],            **text_feat,            **patterns        })        # Fortschrittsanzeige        if (idx + 1) % 5000 == 0:            print(f"   {idx+1:,}/{total:,} verarbeitet...")    print(f" {len(results):,} Kommentare analysiert")    return pd.DataFrame(results)def aggregate_by_issue(features_df):    """Aggregiert NLP-Features pro Issue."""    print(" Aggregiere pro Issue...")    aggregated = features_df.groupby('issueid').agg({        'sentiment_compound': ['mean', 'std', 'min', 'max'],        'sentiment_pos': 'mean',        'sentiment_neg': 'mean',        'word_count': ['mean', 'sum'],        'question_count': 'sum',        'politeness_score': 'sum',        'urgency_score': 'sum',        'technical_score': 'sum',        'solution_score': 'sum'    }).reset_index()    # columns umbenennen    aggregated.columns = ['_'.join(col).strip('_') for col in aggregated.columns]    print(f" {len(aggregated):,} Issues aggregiert")    return aggregated

##  Execution

In [2]:
print("="*50)print(" NLP-ANALYSE")print("="*50)# Load utterancesdata_path = Path("data/raw/sample_utterances.csv")if data_path.exists():    utterances = pd.read_csv(data_path)    print(f" Loaded: {len(utterances):,} Kommentare")    # Verarbeiten    features = process_utterances(utterances)    # Aggregieren    issue_features = aggregate_by_issue(features)    # Saven    output_path = Path("data/processed/nlp_features.csv")    output_path.parent.mkdir(parents=True, exist_ok=True)    issue_features.to_csv(output_path, index=False)    print(f"\n Saved: {output_path}")    # Statistiken    if 'sentiment_compound_mean' in issue_features.columns:        avg_sentiment = issue_features['sentiment_compound_mean'].mean()        print(f"\n Durchschnittliches Sentiment: {avg_sentiment:.3f}")else:    print(" Utterances-Datei not found!")

================================================== NLP-ANALYSE================================================== Utterances-Datei not found!